In [1]:
# Setup dataset
import numpy as np
from dynamics_testing import sim, make_train, attractor_dist

SEG, SETTLE, WARM_TRAIN = 2000, 1000, 200
DT = 0.07                              # frame = 0.07 time units (~10 frames per lobe oscillation)
R_CHAOS, R_SADDLE = 24.5, 23.5         # chaotic attractor | chaotic saddle, collapses to C+/- after a random time
N_PRE = N_POST = 2500                  # switch at frame 2500 (T_SW in the detection cell)

def rhs(_, y, r, s=10.0, b=8 / 3):     # time rescaled by DT so dynamics_testing's unit sampling = one frame
    x, yy, z = y
    return [DT * s * (yy - x), DT * (x * (r - z) - yy), DT * (x * yy - b * z)]

LONG = sim(rhs, R_CHAOS, 50_000, (1.0, 1.0, 20.0))
MU, SD = LONG.mean(axis=1, keepdims=True), LONG.std(axis=1, keepdims=True)
norm = lambda y: (y - MU) / SD         # unit variance, so attractor_dist's |a| > 10 check stays meaningful

def collapse_index(yn, win=int(5 / DT), tol=0.05):
    s = np.lib.stride_tricks.sliding_window_view(yn[0], win).std(axis=1)
    idx = np.flatnonzero(s < tol)
    return int(idx[0]) if len(idx) else None

TRAIN = norm(make_train(rhs, SEG, b1=R_CHAOS, b2=R_SADDLE, init=(1.0, 1.0, 20.0)))
assert collapse_index(TRAIN) is None, "saddle segment collapsed during training; change init"

DIRS = {"chaos->saddle": (R_CHAOS, R_SADDLE), "saddle->chaos": (R_SADDLE, R_CHAOS)}

def make_streams(n_ics, seed):
    rng = np.random.default_rng(seed)
    streams, collapse = {d: [] for d in DIRS}, {d: [] for d in DIRS}
    for d, (r0, r1) in DIRS.items():
        while len(streams[d]) < n_ics:
            x = norm(make_train(rhs, N_PRE, b1=r0, b2=r1, n_seg=2, init=rng.uniform([-15, -15, 5], [15, 15, 40])))
            c = collapse_index(x)
            if c is not None and c < N_PRE:          # saddle died before the switch: resample
                continue
            streams[d].append(x); collapse[d].append(c)
    return streams, collapse

STREAMS, COLLAPSE = make_streams(4, seed=700)

ch, sd = norm(LONG[:, :4000]), norm(sim(rhs, R_SADDLE, 2000, (1.0, 1.0, 20.0)))
print("chaos vs chaos  :", round(attractor_dist(ch[:, :2000], ch[:, 2000:]), 3))
print("chaos vs saddle :", round(attractor_dist(ch[:, :2000], sd), 3))     # expect ~ chaos vs chaos: subtle switch
print("collapse frame after switch (None = stayed on saddle):", COLLAPSE)

chaos vs chaos  : 0.087
chaos vs saddle : 0.183
collapse frame after switch (None = stayed on saddle): {'chaos->saddle': [None, None, None, None], 'saddle->chaos': [None, None, None, None]}


In [2]:
import time
import pandas as pd
from tqdm import tqdm
from testing_harness import RN
from spike_sharing import SpikingNetwork

SEEDS = list(range(8))
ALPHAS = [1e-8, 1e-5, 1e-3, 1e-2, 1e-1, 1e0]
RN_BASE = dict(input_dims=3, spectral_radius=0.6, input_weight=1.2, average_degree_nodes=10, num_nodes=100,
               leakage=0, omega0=0.01, epsilon1=-0.2, epsilon2=0.6, mean_phase_tolerance=0.01, error_tolerance=np.inf)
T_REF0, T_REF1, T_SW = 1000, 2000, N_PRE             # settle | reference | false-alarm window | switch
TAU_S, PERSIST, TAU_CTX = 20, 10, 60                 # detector EMA | persistence | context filter (= spiking mod_tau)
KGRID = np.round(np.arange(1, 12.01, 0.25), 2)
HOLDOUT = norm(LONG[:, 10_000:12_000])               # chaotic segment not in TRAIN, for ESN alpha

def ema(s, tau):
    out, m = np.empty_like(s), s[0]
    for i, v in enumerate(s):
        m += (v - m) / tau; out[i] = m
    return out

def detect(s, k):
    s = ema(np.asarray(s, float), TAU_S)
    mu, sd = s[T_REF0:T_REF1].mean(), s[T_REF0:T_REF1].std() + 1e-12
    run = np.convolve((np.abs(s - mu) > k * sd).astype(int), np.ones(PERSIST, int))[:len(s)] >= PERSIST
    post = np.flatnonzero(run[T_SW:])
    return bool(run[T_REF1:T_SW].any()), (post[0] if len(post) else np.nan)

def ref_dist(X):                                      # RMS distance of each row from its reference-window mean
    return np.sqrt(np.mean(np.abs(X - X[T_REF0:T_REF1].mean(axis=0)) ** 2, axis=1))

def rel_phasor(phases):                               # e^{i(theta_i - psi)}, psi = global mean phase
    return np.exp(1j * (phases - np.angle(np.mean(np.exp(1j * phases)))))

def train_esn(seed, w_in):
    esn = RN(**RN_BASE, link_strength_change_ratio=0.0, model_seed=seed); esn.input_weights = w_in
    esn.train(TRAIN, warmup_time=WARM_TRAIN)
    W = {}
    for a in ALPHAS:
        esn.regularization, esn.lsqrs = a, False; esn.compute_weights(); W[a] = esn.output_weights.copy()
    esn.reset_initial_states(seed_offset=1); esn.reset_history()
    for t in range(HOLDOUT.shape[1]):
        esn.advance(HOLDOUT[:, t])
    H = np.asarray(esn.node_states_history)[WARM_TRAIN:-1]
    err = {a: np.mean(np.linalg.norm(H @ W[a].T - HOLDOUT[:, WARM_TRAIN + 1:].T, axis=1)) for a in ALPHAS}
    a = min(err, key=err.get)
    esn.output_weights = W[a]; esn.reset_history()
    return esn, a

SIG, ESN_ALPHA = {}, {}
for seed in SEEDS:
    t0 = time.time()
    spk = SpikingNetwork(model_seed=seed)                                  # final model, c = 1.0, Thomas hyperparameters
    rs = RN(**RN_BASE, link_strength_change_ratio=0.6, model_seed=seed); rs.input_weights = spk.input_weights.copy()
    esn, ESN_ALPHA[seed] = train_esn(seed, spk.input_weights.copy())

    for d, xs in tqdm(STREAMS.items()):
        for i, x in enumerate(xs):
            T = x.shape[1]
            spk.reset_initial_states(1 + i); spk.reset_history()
            rs.reset_initial_states(seed_offset=1 + i); rs.reset_history()
            esn.reset_initial_states(seed_offset=1 + i); esn.reset_history()
            R_spk, R_rs, err = np.empty(T), np.empty(T), np.full(T, np.nan)
            ctx_spk = np.empty((T, spk.num_links), np.complex64)
            ctx_rs = np.empty((T, rs.num_links), np.complex64)
            st_esn = np.empty((T, esn.num_nodes))
            z_rs, n_lp = None, None
            for t in range(T):
                spk.advance(x[:, t], save_history=False)
                rs.advance(x[:, t], save_history=False)
                esn.advance(x[:, t], save_history=False)
                R_spk[t] = np.abs(np.mean(np.exp(1j * spk.link_phase)))
                R_rs[t] = np.abs(np.mean(np.exp(1j * rs.link_states)))
                ctx_spk[t] = spk.rel_lp
                r = rel_phasor(rs.link_states)
                z_rs = r if z_rs is None else z_rs + (r - z_rs) / TAU_CTX
                ctx_rs[t] = z_rs
                n_lp = esn.node_states.copy() if n_lp is None else n_lp + (esn.node_states - n_lp) / TAU_CTX
                st_esn[t] = n_lp
                if t + 1 < T:
                    err[t] = np.linalg.norm(esn.output_weights @ esn.node_states - x[:, t + 1])
            err[-1] = err[-2]
            SIG[(seed, d, i)] = {"spike_R": R_spk, "spike_ctx": ref_dist(ctx_spk),
                                 "RS_R": R_rs, "RS_ctx": ref_dist(ctx_rs),
                                 "ESN_err": err, "ESN_state": ref_dist(st_esn)}
    print(f"seed {seed}: {time.time() - t0:.0f}s | ESN alpha {ESN_ALPHA[seed]:g}", flush=True)

pd.to_pickle(dict(SIG=SIG, COLLAPSE=COLLAPSE, ESN_ALPHA=ESN_ALPHA), "lorenz_detection_signals.pkl")

# ======================= analysis: matched false-alarm rate =======================
g = pd.DataFrame([dict(seed=s, dir=d, ic=i, detector=n, k=k, **dict(zip(("fa", "lat"), detect(v, k))))
                  for (s, d, i), sig in SIG.items() for n, v in sig.items() for k in KGRID])
g["collapse"] = pd.Series([COLLAPSE[d][i] for d, i in zip(g.dir, g.ic)], dtype="float64")
g["before_collapse"] = g.collapse.isna() | (T_SW + g.lat < g.collapse)
g["hit"] = ~g.fa & g.lat.notna() & g.before_collapse
g.to_csv("lorenz_detection.csv", index=False)

far = g.groupby(["detector", "k"]).fa.mean()                            # pooled over directions
K = {n: f[f <= 0.05].index.get_level_values("k").min() for n, f in far.groupby(level="detector")}
print("k at <=5% FA:", K)                                               # NaN = no operating point
m = g[g.k == g.detector.map(K)]
res = m.groupby(["dir", "detector"]).agg(fa=("fa", "mean"), hit=("hit", "mean"))
res["lat_median"] = m[m.hit].groupby(["dir", "detector"]).lat.median()
res["lat_iqr"] = m[m.hit].groupby(["dir", "detector"]).lat.agg(lambda v: v.quantile(.75) - v.quantile(.25))
print(res.round(2).to_string())
print("detectors with no 5% operating point:", [n for n, k in K.items() if np.isnan(k)])

100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [29:35<00:00, 887.98s/it]

seed 0: 1787s | ESN alpha 1e-05



100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [29:05<00:00, 872.86s/it]

seed 1: 1757s | ESN alpha 1e-08



100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [28:50<00:00, 865.09s/it]

...


100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [28:50<00:00, 865.15s/it]

seed 6: 1741s | ESN alpha 1e-05



100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [28:46<00:00, 863.34s/it]

seed 7: 1738s | ESN alpha 1e-08


k at <=5% FA: {'ESN_err': np.float64(7.75), 'ESN_state': np.float64(4.75), 'RS_R': np.float64(1.0), 'RS_ctx': np.float64(1.0), 'spike_R': np.float64(2.5), 'spike_ctx': np.float64(4.75)}
                           fa   hit  lat_median  lat_iqr
dir           detector                                  
chaos->saddle ESN_err    0.09  0.00         NaN      NaN
              ESN_state  0.00  0.22       997.0    10.00
              RS_R       0.00  0.28       935.0    35.00
              RS_ctx     0.00  0.38       984.5    91.50
              spike_R    0.00  0.28      1136.0   146.00
              spike_ctx  0.00  0.28       961.0   459.00
saddle->chaos ESN_err    0.00  0.44       600.5   142.00
              ESN_state  0.00  0.50       413.0   800.25
              RS_R       0.03  0.22       738.0    23.00
              RS_ctx     0.06  0.22       741.0    20.50
              spike_R    0.03  0.03        25.0     0.00
              spike_ctx  0.09  0.41        31.0   838.00
detectors with n

In [4]:
h = m[m.hit & (m.dir == "chaos->saddle")]
print(h.groupby(["detector", "ic"]).lat.agg(["count", "median"]).unstack("ic"))

          count                median                       
ic            0    1    2    3      0      1       2       3
detector                                                    
ESN_state   7.0  NaN  NaN  NaN  997.0    NaN     NaN     NaN
RS_R        8.0  NaN  NaN  1.0  934.0    NaN     NaN  1409.0
RS_ctx      8.0  NaN  NaN  4.0  978.5    NaN     NaN  1296.5
spike_R     NaN  2.0  6.0  1.0    NaN  954.5  1135.0  1193.0
spike_ctx   8.0  NaN  1.0  NaN  962.0    NaN   511.0     NaN
